In [1]:
import os

import pandas as pd

"""
file path
"""
domain = os.path.dirname(os.path.abspath('__file__'))
print(domain)
FILE_PATH_sample = os.path.join(domain, 'datasets', 'tmp_xx_ontology_sample0.xlsx')
FILE_PATH_behave = os.path.join(domain, 'datasets', 'tmp_xx_ontology_sample_history_behave.xlsx')
FILE_PATH_history_plc = os.path.join(domain, 'datasets', 'tmp_xx_ontology_sample_history_plc.xlsx')
FILE_PATH_profile = os.path.join(domain, 'datasets', 'tmp_xx_ontology_sample_profi.xlsx')
FILE_PATH_qw = os.path.join(domain, 'datasets', 'tmp_xx_ontology_sample_qw.xlsx')

e:\claude-code\Ontology_AgenticML\cpic_datasets


In [2]:
data_sample = pd.read_excel(FILE_PATH_sample)
data_history_plc = pd.read_excel(FILE_PATH_history_plc)
data_profile = pd.read_excel(FILE_PATH_profile)
data_qw = pd.read_excel(FILE_PATH_qw)
data_behave = pd.read_excel(FILE_PATH_behave)

C:\Users\Yuan\AppData\Roaming\Python\Python39\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [3]:
# training_dataset = data_sample[data_sample.insurance_period_beg_dt != 20260430]
# test_dataset = data_sample[data_sample.insurance_period_beg_dt == 20260430]

# training_dataset.to_excel(os.path.join(domain, 'datasets', 'training_dataset.xlsx'), index=False)
# test_dataset.to_excel(os.path.join(domain, 'datasets', 'test_dataset.xlsx'), index=False)

In [4]:
data_sample[data_sample.appnt_id_no == 'c0a5a271fb769282e5945ceea02b0313']['tel_no'].unique()

array(['e665ed1cccbbdf91f13e24f97efbaa60'], dtype=object)

In [5]:
target_sample_appnt_id_lst = data_sample[data_sample.insurance_period_beg_dt == 20260430]['appnt_id_no'].unique().tolist()
target_sample_tel_no_lst = data_sample[data_sample.insurance_period_beg_dt == 20260430]['tel_no'].unique().tolist()

print("预测的目标用户证件号ID数量: ", len(target_sample_appnt_id_lst), "；手机号数量: ", len(target_sample_tel_no_lst))

预测的目标用户证件号ID数量:  461 ；手机号数量:  464


In [6]:
data_profile_target = data_profile[data_profile['tel_no'].isin(target_sample_tel_no_lst)]
data_qw_target = data_qw[data_qw['tel_no'].isin(target_sample_tel_no_lst)]
data_behave_target = data_behave[data_behave['tel_no'].isin(target_sample_tel_no_lst)]
data_history_plc_target = data_history_plc[data_history_plc['appnt_id_no'].isin(target_sample_appnt_id_lst)]

In [7]:
# Compress data_qw_target into one row per tel_no by concatenating content_str
data_qw_target_agg = data_qw_target.sort_values(['tel_no','msg_time']).groupby('tel_no').apply(lambda group: '。'.join(group['content_str'].dropna())).reset_index(name='qw_text')
data_qw_target_agg[data_qw_target_agg['tel_no'] == 'e665ed1cccbbdf91f13e24f97efbaa60']

,tel_no,qw_text
22,e665ed1cccbbdf91f13e24f97efbaa60,我已经添加了你，现在我们可以开始聊天了。。您好，我是您的太平洋专属保险顾问，将为您提供方案规...


In [8]:
# Compress data_behave_target into one row per tel_no by concatenating content_str
data_behave_target_agg = data_behave_target.sort_values(['tel_no','evt_start_tm']).groupby('tel_no').apply(lambda group: '; '.join(group['button_nm'].dropna())).reset_index(name='behave_text')
data_behave_target_agg[data_behave_target_agg['tel_no'] == '0192f69d68d0157cb7fce084258a9f0f']

,tel_no,behave_text
1,0192f69d68d0157cb7fce084258a9f0f,查看有效保单; 保单列表; 首页-五宫格-我的保单; 查看有效保单


In [9]:
# Compress data_history_plc_target into one row per appnt_id_no by concatenating content_str
data_history_plc_target_agg_cvrg = data_history_plc_target.groupby('appnt_id_no').apply(lambda group: '; '.join(group['cvrg_name'].dropna())).reset_index(name='cvrg_name_text')
data_history_plc_target_agg_productbigtype = data_history_plc_target.groupby('appnt_id_no').apply(lambda group: '; '.join(group['productbigtype'].dropna())).reset_index(name='productbigtype_text')
data_history_plc_target_agg_insured_amt = data_history_plc_target.groupby('appnt_id_no').apply(lambda group: '; '.join(group['insured_amt'].astype(str).dropna())).reset_index(name='insured_amt_text')

data_history_plc_target_agg = data_history_plc_target_agg_cvrg.merge(pd.merge(data_history_plc_target_agg_productbigtype, data_history_plc_target_agg_insured_amt, how = 'outer', on = 'appnt_id_no'), how = 'outer', on = 'appnt_id_no')

In [10]:
# 客户数据拼接
test_set = \
data_sample[data_sample.appnt_id_no.isin(target_sample_appnt_id_lst)][['appnt_id_no','tel_no']].drop_duplicates()\
    .merge(data_profile_target, on = 'tel_no', how = 'left')\
    .merge(data_behave_target_agg, on = 'tel_no', how = 'left')\
    .merge(data_qw_target_agg, on = 'tel_no', how = 'left')\
    .merge(data_history_plc_target_agg, on = 'appnt_id_no', how = 'left')\
    .drop_duplicates(subset=['appnt_id_no'])

In [11]:
df_dimension = pd.read_excel(os.path.join(domain, '测试用表字典.xlsx'), sheet_name = '数据字典')

In [12]:
field_dict = df_dimension[['字段名', '中文含义']].set_index('字段名')['中文含义'].to_dict()
print(field_dict)

{'appnt_id_no': '投保人证件号', 'cvrg_cd': '险种代码', 'cvrg_name': '险种名称', 'tel_no': '手机号', 'insurance_period_beg_dt': '责任生效日期', 'policy_no': '保单号', 'main_rider_ind': '是否主险', 'long_short_type_cd': '长短险标志', 'insurance_period_end_dt': '责任失效日期', 'productbigtype': '险种类别', 'productbigtype2': '险种类别2', 'period': '责任期', 'insured_amt': '保额', 'paid_prem': '保费', 'sales_branch': '分公司', 'sales_cen_branch': '中支公司', 'sales_buzi_branch': '支公司', 'atm_valid_policy_ind': '是否有效', 'channel_nm': '渠道', 'business_type': '业务线', 'channel_prnt_type': '渠道类型', 'channel_type': '渠道简称', 'evt_type': '事件类型', 'evt_nm': '事件类型2', 'page_nm': '页面名称', 'button_nm': '按钮名称', 'channel_nm_en': '渠道英文名', 'evt_start_tm': '事件时间', 'customer_id': '企微id', 'chat_id': '对话id', 'content': '消息内容', 'content_str': '消息内容_文字', 'from': '消息发送对象', 'to': '消息接收对象', 'msg_id': '消息id', 'msg_time': '消息时间', 'msg_type': '消息类型', 'gender': '性别', 'age': '年龄', 'edu_degree': '学历', 'mari_stat': '婚姻状态', 'occp': '职业', 'lf_cust_ind': '是否寿险客户', 'pt_cust_ind': '是否产险客户', 'app_

In [13]:
for i in test_set.columns:
    if i not in field_dict.keys():
        print(i)

behave_text
qw_text
cvrg_name_text
productbigtype_text
insured_amt_text


In [14]:
field_dict.update({"behave_text":"APP按钮点击记录","qw_text":"企业微信记录","cvrg_name_text":"历史购买险种记录","productbigtype_text":"历史购买险种类型","insured_amt_text":"历史购买险种保额"})

In [15]:
test_set.columns = [field_dict.get(col, col) for col in test_set.columns]
# test_set.to_excel(os.path.join(domain, 'KnowledgeBase', 'test_set.xlsx'), index=False)

In [16]:
client_full_info = test_set.to_dict('records')

In [17]:
client_full_info[0]

{'投保人证件号': '015dc06ce5a07309c744c38a4cfd1ffd',
 '手机号': 'a29d89191831949905e6b92cf3b1dc4a',
 '是否给子女投保': 0,
 '是否给配偶投保': 0,
 '是否给父母投保': 0,
 '是否给本人投保': 1,
 '是否有理财类保障': 0,
 '大健康_乳腺癌患者标志': 0,
 ' 大健康_乳腺类疾病患者标志(不含乳腺癌)': 0,
 '城市等级': '新一线城市',
 '城市（优先最近一次ip城市，其次电话号码所在城市）': '杭州',
 '是否长江养老客户': 0,
 '大健康_是否有糖尿病': 0.0,
 '大健康_诊断疾病-前5疾病名称': nan,
 '学历': '未知',
 '大健康_最早一次成功理赔时间': nan,
 '是否营销员': 0,
 '性别': '女性',
 '是否健康险自营客户': 1,
 '大健康_是否有高血压': 0.0,
 '客户收入分层': nan,
 '是否有意外类保障': 0,
 '是否有车险保障': 0,
 '是否有寿险类保障': 0,
 '是否有医疗类保障': 1,
 '是否有家财险保障': 0,
 '是否有重疾类保障': 0,
 '是否活跃（是否最近一个月app或官网或e服务或活动活跃）': 0,
 '是否APP话题互动型用户': 0,
 '是否内容偏好用户': 0,
 '是否汽车内容偏好用户': 0,
 '是否健养内容偏好用户': 0,
 '是否金融内容偏好用户': 0,
 '是否综合内容偏好用户': 0,
 '是否业务手动黑名单客户': 0,
 '监管投诉客户': 0,
 '司内投诉客户': 0,
 '是否太保失效老客户': 0,
 '最近一张保单失效时间': '2023-07-30 00:00:00',
 '最近一次登录app城市': '杭州',
 '最近一次登录app省份': '浙江',
 '最近一次服务购买时间': nan,
 '是否寿险客户': 1,
 '大健康_最近一次乳腺癌理赔时间': nan,
 '大健康_最近一次理赔所对应的产品名称': nan,
 '大健康_最近一次理赔所对应的产品险种': nan,
 '大健康_最近一次成功理赔时间': nan,
 '婚姻状态': '未说明的婚姻状况',
 '婚育状态': 

In [18]:
user_prompt = """
你是一位资深的保险营销策略专家，擅长从多源异构数据中提炼客户洞察，并利用客户画像与行为模式，结合相似客群的成交偏好，进行个性化保险产品推荐。

## 任务目标

结合该客户信息（包括画像标签、APP行为、企微对话、历史保单等），预测其下一步最可能购买的3款保险产品，并给出置信度与推理依据。客户信息将以JSON格式提供，包含但不限于以下字段：
- 画像标签：年龄、性别、职业、收入水平、家庭结构等。
- APP行为：浏览过的险种页面、点击过的按钮等。
- 企微对话：与客户的沟通记录，可能透露需求、兴趣点等。
- 历史保单：客户过往购买的保险产品及其类型、保额等信息。

## 核心约束（必须严格遵守）

1. **协同个性化预测**：推荐必须基于提供的客户自身画像、行为、对话等信息进行推断，不得仅依赖规则或通用推荐。
2. **候选保险产品范围**：最终推荐的“险种名”和“险种代码”必须严格来自以下保险产品列表。不得推荐列表以外的任何产品。所推荐的险种名或险种代码不得出现在该表的历史记录中。若客户无历史成交记录，则正常推荐，但不得推荐重复产品。

## 推荐保险产品列表

| 险种代码 | 险种名 | 险种类别 | 责任期中位数 | 单均保费 | 保额保费杠杆率 | 投保年龄 | 核心卖点 | 适合客群 |
|:---|:---|:---|:---|:---|:---|:---|:---|:---|
| 27885500 | 太保颐护金生终身护理保险 | 护理型 | 2912333 | 9023.79797340088 | 22.998717696511 | 18-65周岁 | 关怀金一次性给付+护理金按月领取（最长120个月）；三类护理责任触发覆盖更多失能场景；现金价值稳增长；支持保单贷款、自动垫交、部分减保 | 18-65岁、有长期家庭护理规划、希望有备用护理金应对意外失能的青壮年及中年人群；尤其适合有家族病史或长期工作压力大的家庭支柱，投保门槛友好，缴费方式灵活 |
| 27885200 | 太保岁优福2025终身护理保险 | 护理型 | 2912353 | 3234.18467583497 | 12.7026111651075 | 出生满28天至70周岁 | 首款税优健康险；每年最高享1080元个税抵扣；含10种特定疾病护理金+疾病身故金；终身保障+现金价值稳健增长 | 已参加基本医疗保险且有个人所得税缴纳的客户，可为自己或配偶/子女/父母投保，每年最高2400元个税扣除，收入越高节税越多 |
| 44734500 | 太保鸿寿年年年金保险（分红型） | 理财型 | 10957 | 39882.3529411764 | 0.897714212081254 | 0-80周岁 | 投保年龄最宽（0-80周岁）；保证利益+分红浮动收益 | 临近退休、需锁定养老收入的人群，0-80岁皆可投保，尤其适合60岁以上高龄补养老收入者以及想提前储备子女教育金的年轻父母，54岁以下还能选祝福金 |
| 44734900 | 太保鸿福添年年金保险（分红型） | 理财型 | 18262 | 52532.6585976627 | 0.88966382705726 | 出生满5天至70周岁 | 投保年龄宽至70周岁；保证利益+分红浮动收益；适合长期养老储备 | 适合0-70岁有长期储蓄、养老规划或财富传承需求的人群，尤其希望通过固定年金+分红实现稳健增值的用户；长期持有可锁定稳定性，保障养老生活现金流 |
| 44819500 | 太保鑫享康年（2025）养老年金保险 | 理财型 | 5478 | 48433.6372847011 | 0.930466958831896 | 男性60-80周岁，女性55-80周岁 | 专为中老年设计；男性60-80岁/女性55-80岁可投；缴费期短（可趸交/3年/5年）；第1年末即可领取养老金；含全残保障 | 女性55-80岁、男性60-80岁高龄人群，无需复杂健康告知即可投保；适用于已经退休或即将退休、想补充社会养老缺口的老年人；保障期10/15年内满期即回本，还能早期提前看到现金流 |
| 44733700 | 太保鑫福岁优年金保险（分红型） | 理财型 | 8765 | 11694.6012269938 | 0.986430037806059 | 最高80周岁 | 分红型年金；养老金最早60岁领取，可选领20年/30年；支持年领或月领；保证利益+分红浮动收益 | 0-80岁有品质养老、产服融合需求的中高端人群，尤其看重太保家园养老社区优先入住权益；60岁以上可实现即交即领，最早60岁领取养老金并提供20年或30年抵御长寿风险 |
| 44656800 | 太保盛世鸿运终身寿险（分红型） | 寿险型 | 2912357 | 35469.0854901006 | 3.54300360443091 | 男性0-64周岁，女性0-68周岁 | 保额每年按1.75%复利递增；分红机制提供浮动收益；支持1-2名被保人；含意外豁免保费权益 | 中高净值家庭家庭支柱补充传承保障，尤其40岁以下双职工家庭，可设计为终身现金流；资产传承适合已有足够现金流、对境外资产或遗产规划有需求的高净值人群；双受益人设计对冲风险 |
| 44644700 | 太保盛世传家终身寿险 | 寿险型 | 2912366 | 33533.3612653189 | 3.82551193143896 | 出生满5天至73周岁 | 有效保额每年按2.0%复利递增直至终身；纯固收型增额寿，收益100%确定；含身故+全残双重保障；可保单贷款、年金转换 | 有长期保障、希望为家人提供身故保障或进行财富传承的人群；投保5天-73岁范围极广，尤其适合高净值家庭晚年规划传承以及想锁定长期收益的中产以上家庭 |
| 87230100 | 乐享百万医疗保险（H2019） | 医疗型 | 364 | 1202.01081562645 | 3318.92537778915 | 0-100周岁 | 重疾0免赔；含住院垫付服务；重症病房补贴500元/天；一般医疗100万+特疾200万+重疾400万；不限社保用药 | 首次投保最高65岁；追求高额医疗费用报销、需重疾0免赔保障的成年人，尤其适合60岁以上老年人首次投保，但60岁以上需提前确认是否免体检 |
| 27853200 | 太保特药保2.0医疗保险 | 医疗型 | 364 | 98.1823502457718 | 30555.3899707059 | 出生满28天至100周岁 | 300万特药保额；含110种恶性肿瘤特药+5种指定疾病特药+20种博鳌境外进口特药，共计135种境内外特药保障 | 对癌症等重大疾病有顾虑、希望获得高额特药保障的人群；出生28天至100岁均可投保，尤其适合看重全球进口癌症靶向药及CAR-T保障的家庭；50岁以下人群费率较低 |
| 87011100 | 心安·怡住院费用医疗保险（H2017A） | 医疗型 | 364 | 1205.73484597914 | 178.187303256483 | 0-100周岁 | 住院0免赔；保障四档可选2-20万；88种重疾住院保额翻倍；不限社保目录用药；可续保至80周岁 | 预算充足、追求医疗0免赔且偏爱太平洋人寿品牌的人群；健康情况良好希望零免赔疾病住院保障，尤其适合四五十岁中产阶级，搭配社会医保实现住院全覆盖 |
| 87300200 | 太保安享百万2025医疗保险（费率可调） | 医疗型 | 5478 | 627.271619388598 | 6266.94954927246 | 出生满28天至85周岁 | 15年长期保障；投保年龄宽至85周岁；保障期满可申请重新投保 | 追求15年稳定医疗保障的人群；为高龄父母（最高85岁）寻求长期医疗保障的子女，尤其适合老年人做好医疗费预算，使老年重疾治疗有底 |
| 87220200 | 安享宝贝少儿长期医疗保险（费率可调） | 医疗型 | 6208 | 330.058724021266 | 151.488193951747 | 出生满15天至12周岁 | 保至18周岁的少儿长期医疗；住院医疗+意外门急诊；有社保年保费仅456元起；低免赔门槛 | 0-12岁儿童，看重长期稳定医疗保障、希望为孩子储备持续到18岁的健康保障的家庭；解决小额医疗费用问题 |
| 11331100 | 个人意外伤害保险 | 意外型 | 364 | 21.9006009123282 | 2209.9971651436 | 16-65周岁 | 意外身故+伤残；180日内因意外身故或残疾给付保险金 | 65岁以下所有人群，尤其适合户外工作或经常出行人群；覆盖核心意外死亡与伤残保障，投保门槛低 |
| 11430100 | 世纪行人身意外伤害保险（C款） | 意外型 | 364 | 3.05309528713929 | 351840.680545401 | 出生满28天至70周岁 | 投保年龄宽泛（0-70周岁）；身故+残疾保障；含新冠肺炎扩展保障；投保简便、保费实惠 | 0-70岁全年龄段人群；不限职业，尤其适合乘坐各种公共交通工具、自驾出行或经常外出的上班族；追求高性价比意外险的普通工薪家庭 |
| 26621100 | 疫苗接种意外伤害保险 | 意外型 | 364 | 2.68314670779148 | 82121.3317689721 | 全年龄段 | 保障疫苗接种意外身故/伤残；覆盖接种后不良反应导致的各种意外伤害 | 为婴幼儿、儿童接种国家计划内及计划外疫苗的家庭，和在疫情防控期间接种加强疫苗的成年人；为全家接种时加一个意外底保，保费低廉 |
| 73834200 | 太保爱心保（尊享2025）重大疾病保险 | 重疾型 | 2912351 | 4317.12591785415 | 38.9817824006066 | 0-65周岁 | 120种重疾+20种中症+40种轻症保障；保障期限灵活（保至60/70/80周岁或终身）；交费期内初次罹患特定疾病可豁免后续保费；含保单贷款 | 预算有限、为大额保额做备用减轻医疗压力、有既往病史或纯粹加保需要的人群；可选定期至60/70/80岁，且最高65岁可投，重疾年轻化趋势下的工薪家庭 |
| 44661500 | 太保金生无忧2025（成人版）重大疾病保险 | 重疾型 | 2912353 | 5140.38943569553 | 57.4453456567832 | 男性18-62周岁，女性18-65周岁 | 重疾保障至300%基本保额；含重疾关爱金及特定重疾翻倍赔付；搭配"太保蓝本·无忧管家"一站式重疾专案管理服务 | 18-62岁（男）/18-65岁（女）有重疾保障需求的中青年，看重保额可达300%的高杠杆，追求重点疾病翻倍赔付，也可作为对太保绿通管理服务看重的人群 |
| 44661600 | 太保金生无忧2025（少儿版）重大疾病保险 | 重疾型 | 2912353 | 4713.0832999064 | 102.249983332931 | 出生满28天至17周岁 | 重疾保险金+重疾关爱金+特定重疾保险金可叠加理赔，最高3倍重疾保额；储蓄型保险，身故赔保额；可附加多次重疾及轻症保障 | 0-17岁未成年人群的家长，希望配置覆盖少儿和成人阶段多重高发重疾、保障终身的优质储蓄型重疾险；18-60岁期间确诊重疾可额外叠加赔付，孩子成长全阶段风险覆盖更全 |

## 输出要求

以合法JSON格式返回，确保已严格排除客户已购产品。结构如下：
```
{
  "投保人证件号": [
    {
      "rank": 1,
      "险种名": "如实填写（来自product_catalog，且未曾购买）",
      "险种代码": "如实填写",
      "confidence": 0.xx,
      "reason": "基于[客户自身信号]与[相似客群偏好]的综合推理，例如：该客户无该险种购买记录，画像为年轻父母，浏览过少儿险页面，且相似画像客群中少儿险成交率最高，故推荐..."
    },
    ...
  ]
}
```
- `rank` 为1~3，按购买可能性降序排列。
- `confidence` 为0~1的小数，综合数据完整度与相似客群强度给出。
- `reason` 必须体现如何结合客户个体信号与相似客群偏好，并解释为何认定为未购买过。

## 注意

如果无法实现预测，请返回如下JSON，并注明无法预测的原因，不得随意编造
```
{
  "投保人证件号": "无法预测的原因"
}
```
"""

In [19]:
import json
import random
import re

from openai import OpenAI
from tqdm import tqdm

result_clients = []
prompt_log = []

client = OpenAI(
    base_url="https://api.minimaxi.com/v1",
    api_key="sk-api-1LdPFZtlhNK-FCngiza_LAImvEbjE472xuDSVYlhIlwgJ-J_y_-nYgzRFqY_l0vyIi_BnvLRlR95d9kWpfyYn24cf4iXtyS9a2ldsItMS5YreuW4x5dHPw4",
)

random.seed(42)
client_sample = random.sample(client_full_info, 3)
# for i in tqdm(client_full_info, desc="推理中"):
for i in tqdm(client_sample, desc="推理中"):

    # 逐一加载客户信息
    client_info = f"""
    客户信息如下：

    {str(i)}"""

    response = client.chat.completions.create(
        model="MiniMax-M2.7",
        messages=[
            {"role": "user", "content": user_prompt + client_info},
        ],
    )
    # print(response.choices[0].message.content)
    content_i = response.choices[0].message.content

    # 匹配 ```json ... ``` 或 ``` ... ```
    pattern = r"```(?:json)?\s*\n?(.*?)```"
    match = re.search(pattern, content_i, re.DOTALL | re.IGNORECASE)
    if match:
        raw = match.group(1).strip()
        result_dct = json.loads(raw)

    prompt_log.append(user_prompt + client_info)
    result_clients.append(result_dct)

推理中: 100%|██████████| 3/3 [06:21<00:00, 127.18s/it]


In [20]:
result_clients

[{'b4a33f8c4608332d2e4581ba1ae6d805': [{'rank': 1,
    '险种名': '太保盛世传家终身寿险',
    '险种代码': '44644700',
    'confidence': 0.78,
    'reason': '该客户61岁已婚女性，已是寿险客户但当前持有寿险缺口保额13万元，寿险类评分10.8表明有一定购买意愿。客户APP行为显示频繁查看保单和续期缴费，说明对保险产品有一定关注度且保险意识强。历史保单显示已购医疗+意外类保障，但无寿险产品，盛世传家终身寿险有效保额每年按2%复利递增，身故+全残双重保障，可进行财富传承与长期保障规划。相似客群中61岁左右有成年子女的已婚客户对增额终身寿的传承功能偏好显著。'},
   {'rank': 2,
    '险种名': '太保鑫享康年（2025）养老年金保险',
    '险种代码': '44819500',
    'confidence': 0.72,
    'reason': '该客户61岁女性（55-80周岁可投保范围），子女已35岁成年，家庭责任相对减轻，但理财类评分7.1且存在66000元理财缺口保费，说明有养老储备需求。客户已持有医疗和意外保障，配置优先级上适合补充养老年金。该产品缴费期短（趸/3/5年可选），第1年末即可领取养老金，能够快速补充社会养老缺口。相似画像客群中临近退休且有理财缺口的客户对早期领取型养老年金偏好明显，尤其产品还有全残保障设计。'},
   {'rank': 3,
    '险种名': '太保爱心保（尊享2025）重大疾病保险',
    '险种代码': '73834200',
    'confidence': 0.65,
    'reason': '该客户61岁但当前无任何重疾类保障（重疾缺口保额-50000元），重疾类评分7.7虽不算高，但考虑到年龄增长带来的健康风险提升，重疾保障缺口实际存在。该产品0-65周岁可投保，保障期限灵活可选（至60/70/80/终身），60岁以上保费虽偏高但仍可接受。客户已有医疗险（乐享百万）可覆盖医疗费用，重疾险作为收入补偿和康复费用补充。相似客群中61岁无重疾保障的客户倾向于先配置基础重疾保障，且产品支持保单贷款功能。'}]},
 {'21fb384f12a448616276